# Instead of making python OCR which doesnt gurantee to extract everything correctly we will use genAI model that will extract and give result in desired format

In [19]:
"""
medical_ocr.py
Classic OCR (NO GenAI)
- Image or PDF input
- Handles blocks, tables, paragraphs (best-effort)
- Extracts patient info + lab values
- Outputs a Python dictionary
"""

import cv2
import pytesseract
import numpy as np
from PIL import Image
import re
import os
from pdf2image import convert_from_path

# ---------- CONFIG ----------
# If Tesseract is not in PATH, uncomment and set this:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# ---------- IMAGE PREPROCESSING ----------
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu thresholding for clean text
    thresh = cv2.threshold(
        blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    # Optional: deskew
    coords = np.column_stack(np.where(thresh > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = thresh.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    deskewed = cv2.warpAffine(
        thresh, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )

    return deskewed

# ---------- OCR ----------
def ocr_text_from_image(image_path):
    processed = preprocess_image(image_path)
    config = "--oem 3 --psm 6"  # LSTM engine, block of text
    text = pytesseract.image_to_string(processed, config=config)
    return text

# ---------- PDF TO IMAGES ----------
def pdf_to_images(pdf_path, dpi=300):
    pages = convert_from_path(pdf_path, dpi=dpi)
    image_paths = []
    for i, page in enumerate(pages):
        out = f"_page_{i}.png"
        page.save(out, "PNG")
        image_paths.append(out)
    return image_paths

# ---------- REGEX HELPERS ----------
def find(pattern, text):
    m = re.search(pattern, text, re.IGNORECASE)
    return m.group(1).strip() if m else None

# ---------- PARSE MEDICAL FIELDS ----------
def parse_medical_report(text):
    def find(pattern):
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(1).replace(",", ".").strip() if m else None

    data = {
        "patient_details": {
            "name": find(r"Del:\s*([A-Z ]+)"),
            "sex": find(r"Sesso:\s*(Male|Female|M|F)"),
            "date_of_birth": find(r"Data di Nascita:\s*([\d/.\-]+)")
        },

        "cbc": {
            "wbc": find(r"\bWBC\s+([\d.,]+)"),
            "rbc": find(r"\bRBC\s+([\d.,]+)"),
            "hemoglobin": find(r"\bHGB\s+([\d.,]+)"),
            "hematocrit": find(r"\bHCT\s+([\d.,]+)"),
            "mcv": find(r"\bMCV\s+([\d.,]+)"),
            "mch": find(r"\bMCH\s+([\d.,]+)"),
            "mchc": find(r"\bMCHC\s+([\d.,]+)"),
            "platelets": find(r"\bPLT\s+([\d.,]+)")
        },

        "biochemistry": {
            "glucose": find(r"S-GLUCOSIO\s+([\d.,]+)"),
            "urea": find(r"S-UREA\s+([\d.,]+)"),
            "uric_acid": find(r"S-URATO\s+([\d.,]+)"),
            "creatinine": find(r"S-CREATININA\s+([\d.,]+)")
        }
    }

    return data


# ---------- MAIN PIPELINE ----------
def process_medical_file(path):
    combined_text = ""

    if path.lower().endswith(".pdf"):
        images = pdf_to_images(path)
        for img in images:
            combined_text += "\n" + ocr_text_from_image(img)
            # cleanup temp images
            try:
                os.remove(img)
            except OSError:
                pass
    else:
        combined_text = ocr_text_from_image(path)

    return parse_medical_report(combined_text)

# ---------- RUN EXAMPLE ----------
if __name__ == "__main__":
    # Change this to your file path (image or PDF)
    INPUT_PATH = "C:/Users/91945/Desktop/HealthVison/ml-service/data/Screenshot-2023-04-03-at-1.01.30-PM.png"  # or "medical_report.pdf"

    result = process_medical_file(INPUT_PATH)
    print(result)


{'patient_details': {'name': None, 'sex': None, 'date_of_birth': None}, 'cbc': {'wbc': '9.04', 'rbc': '463', 'hemoglobin': '13.3', 'hematocrit': None, 'mcv': '88.8', 'mch': '28.7', 'mchc': '324', 'platelets': '286'}, 'biochemistry': {'glucose': '98', 'urea': '28', 'uric_acid': None, 'creatinine': None}}


In [17]:
import pytesseract
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print(pytesseract.get_tesseract_version())


5.5.0.20241111
